# Self-Healing Feeder: Two-Tier Edge HIF-Sensor -> Graph FLISR (200-400 ohm & Full Range)

Runs the pre-registered two-tier fusion study (`research/hif-fusion/PLAN.md`) on Kaggle GPU
(offline Linux OpenDSS wheels + PyTorch CUDA GPU + frozen `model_int8.npz` from `christoopherchau/ch-y-n`):
1. Loads the frozen Tier-1 Edge HIF-Sensor (`417` params / `504` bytes int8 Tiny MLP, 4th-order Butterworth
   anti-alias front end at `fc = 2,047.9 Hz` Python / `2,045.1 Hz` LTspice, 12-bit `15.36 kHz` ADC,
   `16` window features over `K = 8` cycles, `PERSIST = 3`).
2. Runs OpenDSS sweeps + Tier-1 Emanuel arc synthesis across **5 arc surfaces** and **7 background disturbances**
   on IEEE 123-bus and unseen IEEE 34-bus (passive + active reconfigurable/looped 0-50% DER grids).
3. Evaluates paired ablations (`60Hz_only` vs `60Hz_plus_edge`) on `PerNodeGBDT`, `ZoneTreeGNN`,
   `ActivePerNodeGBDT`, and bidirectional edge-aware `ActiveGridGATv2` (+ `diff_p_hif` edge feature).
4. Evaluates Split Conformal Prediction (`aps_rand` primary at 90%/95% with `U_SEED = 12345`, plus `lac`)
   and scores pre-committed pass/fail criteria `H1-H4`.
5. Packages isolated outputs into `/kaggle/working/hif_fusion_results.zip` without touching any legacy results.

Simulation only. Nothing here is field data.

In [ ]:
import hashlib, os, shutil, zipfile
from pathlib import Path

DATASET_DIR = "/kaggle/input"
WORK = "/kaggle/working/self-healing-feeder"
if os.path.exists(WORK):
    shutil.rmtree(WORK)

src = next((root for root, _dirs, files in os.walk(DATASET_DIR) if "run.py" in files), None)
if src is None:
    zips = [os.path.join(r, f) for r, _d, fs in os.walk(DATASET_DIR) for f in fs if f.endswith(".zip")]
    for z in zips:
        zipfile.ZipFile(z).extractall("/kaggle/working/_bundle")
    src = next((root for root, _dirs, files in os.walk("/kaggle/working/_bundle") if "run.py" in files), None)
if src is None:
    raise SystemExit("Could not find project bundle in /kaggle/input")
shutil.copytree(src, WORK)
shutil.rmtree("/kaggle/working/_bundle", ignore_errors=True)
os.chdir(WORK)
print("Working dir:", WORK)
print("Top-level:", sorted(os.listdir(WORK)))

# Locate frozen Tier-1 Edge HIF-Sensor model_int8.npz (from attached kernel christoopherchau/ch-y-n)
model_hits = sorted(Path(DATASET_DIR).rglob("model_int8.npz"))
MODEL_NPZ = str(model_hits[0]) if model_hits else None
if MODEL_NPZ:
    sha = hashlib.sha256(Path(MODEL_NPZ).read_bytes()).hexdigest()
    print("Found frozen model_int8.npz:", MODEL_NPZ, "| sha256:", sha)
else:
    print("WARNING: model_int8.npz not found in /kaggle/input; script will use deterministic seed=42 fallback.")

In [ ]:
# Install OpenDSS offline from bundled Linux wheels (no Internet required).
import subprocess, sys
if os.path.isdir(os.path.join(WORK, "wheels")):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index",
                    "--find-links", "wheels", "OpenDSSDirect.py==0.9.4"], check=True)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "OpenDSSDirect.py==0.9.4"], check=True)

import torch, opendssdirect as dss
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("OpenDSS:", dss.Basic.Version())
print("CPU count:", os.cpu_count())

In [ ]:
# Fast verification tests on Kaggle environment before running the full experiment.
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_hif_fusion.py"], check=True)

In [ ]:
# Run the full Two-Tier Edge HIF-Sensor -> Graph FLISR study on Kaggle GPU.
cmd = [sys.executable, "-u", "scripts/hif_fusion_run.py"]
if MODEL_NPZ:
    cmd += ["--model-npz", MODEL_NPZ]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Package isolated results/hif_fusion, provenance/hif_fusion.json, and data/generated/hif_fusion_*.npz.
import json
from pathlib import Path

OUT = Path("/kaggle/working/hif_fusion_results")
if OUT.exists():
    shutil.rmtree(OUT)
(OUT / "hif_fusion").mkdir(parents=True)
(OUT / "provenance").mkdir(parents=True)
(OUT / "generated").mkdir(parents=True)

for p in sorted((Path(WORK) / "results" / "hif_fusion").glob("*.json")):
    shutil.copy2(p, OUT / "hif_fusion" / p.name)
for p in sorted((Path(WORK) / "results" / "provenance").glob("hif_fusion*.json")):
    shutil.copy2(p, OUT / "provenance" / p.name)
for p in sorted((Path(WORK) / "data" / "generated").glob("hif_fusion_*.npz")):
    shutil.copy2(p, OUT / "generated" / p.name)

a34 = json.loads((OUT / "hif_fusion" / "active_ieee34.json").read_text(encoding="utf-8"))
os.chdir("/kaggle/working")
shutil.rmtree(WORK)
shutil.make_archive("/kaggle/working/hif_fusion_results", "zip", OUT)

print("=== Pre-committed H1-H4 criteria summary ===")
print(json.dumps({
    k: v.get("verdict") for k, v in a34["pre_committed_criteria"].items()
}, indent=2))
print("Saved /kaggle/working/hif_fusion_results.zip")